# Phase 4: Silver Layer Construction – Data Cleaning & Standardization

This notebook reads the raw Bronze table, applies cleaning rules based on Phase 3 findings,
and writes a clean, query‑ready Silver Delta table to MinIO.

In [13]:
from pyspark.sql import SparkSession

# Create Spark session with explicit S3/MinIO and Delta Lake configurations
spark = SparkSession.builder \
    .appName("Phase4_SilverConstruction") \
    .master("spark://spark-master:7077") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

print("Spark ready for Silver layer processing.")

Spark ready for Silver layer processing.


## 1. Load Bronze Data

Read the raw Delta table from the Bronze bucket.

In [14]:
# Path to Bronze Delta table
bronze_path = "s3a://ecommerce-bronze/delta/bronze/online_retail"

# Load
df_bronze = spark.read.format("delta").load(bronze_path)

print(f"Bronze rows: {df_bronze.count():,}")
df_bronze.printSchema()

# Show first 5 rows to verify data
df_bronze.show(5, truncate=False)

Bronze rows: 541,909
root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: string (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: integer (nullable = true)
 |-- Country: string (nullable = true)
 |-- ingestion_time: timestamp (nullable = true)

+---------+---------+-----------------------------------+--------+--------------+---------+----------+--------------+--------------------------+
|InvoiceNo|StockCode|Description                        |Quantity|InvoiceDate   |UnitPrice|CustomerID|Country       |ingestion_time            |
+---------+---------+-----------------------------------+--------+--------------+---------+----------+--------------+--------------------------+
|536365   |85123A   |WHITE HANGING HEART T-LIGHT HOLDER |6       |12/1/2010 8:26|2.55     |17850     |United Kingdom|2026-06-12 08:48:53.736991|
|536365   |7

## 2. Remove Invalid Records

Based on Phase 3 analysis:
- Drop rows with negative or zero `UnitPrice` (2 negative + 2,515 zero).
- Drop rows where `Quantity` ≤ 0 *and* the invoice is not a cancellation.
  (We keep cancellation invoices for possible net‑sales calculations.)

In [15]:
from pyspark.sql.functions import col

# 1. Drop rows with UnitPrice <= 0
df_clean = df_bronze.filter(col("UnitPrice") > 0)
removed_price = df_bronze.count() - df_clean.count()
print(f"Removed {removed_price} rows with UnitPrice <= 0.")

# 2. Remove negative/zero Quantity that are NOT cancellations
#    Cancellations have InvoiceNo starting with 'C' (case-insensitive)
df_clean = df_clean.filter(
    (col("Quantity") > 0) |
    (col("InvoiceNo").rlike("^[Cc]"))
)
removed_qty = df_bronze.filter(col("UnitPrice") > 0).count() - df_clean.count()
print(f"Removed {removed_qty} rows with non-cancellation Quantity <= 0.")
print(f"Rows after removal: {df_clean.count():,}")

Removed 2517 rows with UnitPrice <= 0.
Removed 0 rows with non-cancellation Quantity <= 0.
Rows after removal: 539,392


## 3. Handle Missing CustomerID

Instead of dropping ~25% of the data, we replace NULL `CustomerID` with -1,
allowing downstream analysts to filter or group by known vs unknown customers.

In [16]:
from pyspark.sql.functions import coalesce, lit

# Replace NULL CustomerID with -1
df_clean = df_clean.withColumn("CustomerID", coalesce(col("CustomerID"), lit(-1)))

null_cust_after = df_clean.filter(col("CustomerID") == -1).count()
print(f"Rows with unknown CustomerID (set to -1): {null_cust_after:,}")

Rows with unknown CustomerID (set to -1): 132,603


## 4. Remove Exact Duplicates

In [17]:
total_before_dedup = df_clean.count()
df_clean = df_clean.dropDuplicates()
duplicates_removed = total_before_dedup - df_clean.count()
print(f"Total_before_dedup: {total_before_dedup}")
print(f"Exact duplicates removed: {duplicates_removed}")
print(f"Rows after deduplication: {df_clean.count():,}")

Total_before_dedup: 539392
Exact duplicates removed: 5263


[Stage 163:============================>                            (1 + 1) / 2]

Rows after deduplication: 534,129


## 5. Standardize InvoiceDate

The raw `InvoiceDate` is a string like `"12/1/2010 8:26"`.
We convert it to a proper timestamp type and add a dedicated `InvoiceDate_clean` column.

In [18]:
from pyspark.sql.functions import to_timestamp

# Convert string to timestamp (adjust format if needed – here it is month/day/year hour:minute)
df_clean = df_clean.withColumn(
    "InvoiceDate_clean",
    to_timestamp(col("InvoiceDate"), "M/d/yyyy H:mm")
)

# Show a few converted values
df_clean.select("InvoiceDate", "InvoiceDate_clean").show(5, truncate=False)

[Stage 171:============================>                            (1 + 1) / 2]

+---------------+-------------------+
|InvoiceDate    |InvoiceDate_clean  |
+---------------+-------------------+
|12/1/2010 10:47|2010-12-01 10:47:00|
|12/1/2010 11:45|2010-12-01 11:45:00|
|12/1/2010 11:45|2010-12-01 11:45:00|
|12/1/2010 11:49|2010-12-01 11:49:00|
|12/1/2010 12:15|2010-12-01 12:15:00|
+---------------+-------------------+
only showing top 5 rows



## 6. Write Silver Delta Table

The cleaned data is saved as a Delta table in MinIO bucket `ecommerce-silver`.
(You must create this bucket via MinIO Console first.)

In [20]:
# Path for Silver table (bucket must exist)
silver_path = "s3a://ecommerce-silver/delta/silver/online_retail"

# Write as Delta, overwrite mode (idempotent)
df_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .option("delta.columnMapping.mode", "name") \
    .save(silver_path)

print(f"Silver table written to {silver_path}")
print(f"Rows written: {df_clean.count():,}")

Silver table written to s3a://ecommerce-silver/delta/silver/online_retail


[Stage 182:============================>                            (1 + 1) / 2]

Rows written: 534,129


## 7. Verify Silver Table

Read back the Silver table and inspect its schema and a sample of rows.

In [21]:
# Read back
df_silver = spark.read.format("delta").load(silver_path)

print("Silver table schema:")
df_silver.printSchema()

print("Sample rows (5):")
df_silver.show(5, truncate=False)

# Quick row count
print(f"\nTotal rows in Silver: {df_silver.count():,}")

Silver table schema:
root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: string (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: integer (nullable = true)
 |-- Country: string (nullable = true)
 |-- ingestion_time: timestamp (nullable = true)
 |-- InvoiceDate_clean: timestamp (nullable = true)

Sample rows (5):


+---------+---------+---------------------------------+--------+---------------+---------+----------+--------------+--------------------------+-------------------+
|InvoiceNo|StockCode|Description                      |Quantity|InvoiceDate    |UnitPrice|CustomerID|Country       |ingestion_time            |InvoiceDate_clean  |
+---------+---------+---------------------------------+--------+---------------+---------+----------+--------------+--------------------------+-------------------+
|536395   |22867    |HAND WARMER BIRD DESIGN          |48      |12/1/2010 10:47|2.1      |13767     |United Kingdom|2026-06-12 08:48:53.736991|2010-12-01 10:47:00|
|536409   |85116    |BLACK CANDELABRA T-LIGHT HOLDER  |5       |12/1/2010 11:45|2.1      |17908     |United Kingdom|2026-06-12 08:48:53.736991|2010-12-01 11:45:00|
|536409   |22075    |6 RIBBONS ELEGANT CHRISTMAS      |1       |12/1/2010 11:45|1.65     |17908     |United Kingdom|2026-06-12 08:48:53.736991|2010-12-01 11:45:00|
|536412   |22243

In [22]:
# Stop the Spark session when done
spark.stop()